![Platform+](images/banner.png)

# LAB 2: QUERY WITH CoCo
_Hands-on Snowflake for data practitioners_

> **All personas, one lab.** CoCo works the same way regardless of your role — the prompts differ, the mechanics don't. Section 7 has exercises tailored to your persona (analyst, engineer, admin).

👉 **In this lesson, you'll ask business questions in plain English, let CoCo write the SQL, and learn to read and verify it.**

📚 **By the end you'll be able to:**
- Turn a business question into SQL with CoCo
- Read the SELECT / JOIN / GROUP BY pattern behind the answer
- Modify a generated query and verify the result is right
- Discover Snowflake-specific functions by asking CoCo and checking the docs
- Load a skill to give CoCo domain context for your data

**Time:** ~40 min · **Lab interface:** CoCo is your tutor — ask it to explain, write, or fix anything.
*Source data lives in the shared, read-only `PLATPLUS_DB`; the warehouse you run on is your own, named after your username (e.g. `USER_WH`).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('\"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> wh: ' + my_wh + ' | role: ' + my_role)

## Meet CoCo
[CoCo (Cortex Code)](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code) lives in the panel beside your notebook. It's your tutor and your SQL/Python co-author. The loop you'll use all lesson:

1. **Ask** a question in plain English.
2. **Read** the code it generates — don't just run it.
3. **Run** it in a cell.
4. **Modify** and re-ask to refine.
5. **Verify** the result is right — you're accountable, not CoCo.

**Reminder**: Creating with CoCo is an **iterative conversation**.

Working with LLMs and Agents requires time and practice. It is a skill that can be developed, and this is helpful to remember because not every interaction turns out as planned. Sometimes the issue lies on our side and the vague specifications we define in prompts, and other times AI can make assumptions or take wrong turns.

:point_right: **Treat each error as a conversation turn.** 
- The more you iterate, the sharper your prompts become - and the faster you get to working code.

![Expand SQL (image)](images/pp_coco_iterative_process.png)

### Two CoCo UI skills to practice today:
- **Find objects:** ask CoCo and it will list them for you.
- **`@` to reference an object:** type `@` in the CoCo prompt and pick a database, schema, table, or other object. This grounds CoCo on the real column names so its SQL matches your data instead of guessing.

## What is CoCo?

**CoCo** is an AI-driven intelligent agent integrated into the Snowflake platform, and exposed in the Snowsight interface. It is optimized for complex data engineering, analytics, machine learning, and agent-building tasks. It uses an autonomous agent framework to interact directly with your Snowflake environment, with deep understanding of Snowflake's Role-Based Access Control (RBAC), schemas, and best practices.

It can:

- **Generate SQL and Python code** from natural language descriptions
- **Explain existing code** - paste something in and ask "what does this do?"
- **Help with data exploration** - ask it about your tables, stages, and schemas
- **Debug errors** - paste an error message and ask for help

---

### How to open CoCo.

1. Look for the **CoCo icon** in the **right** of your Snowsight window (it looks like a small AI/sparkle icon)
2. **Click it** to open the panel on the right side of the screen

![Open CoCo (image)](images/pp_open_coco_panel.png)

> **Tip:** The CoCo panel opens alongside your notebook. You can type prompts in CoCo while keeping this notebook visible. If you want to open CoCo in a wider view, you can resize the panel by dragging its left edge.

3. The panel has a **message box** at the bottom where you type your prompts
4. Press **Enter** or click the send icon to submit your prompt

![CoCo interface (image)](images/pp_coco_get_started.png)

**CoCo chat interface tour:**

- A. new chat
- B. all chats (history)
- C. close panel
- D. add context / upload
- E. toggle plan mode
- F. model selector
- G. submit (send)

> 💡 **Best practice: Start a fresh chat for each lab.** Click **New chat** (A) when you begin a new lab. Each lab creates its own objects, and a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens. **Do this at the start of every lab from here on.**

> ⚠️ **CoCo runs its own session** - starting with your user's default role, not the notebook's active role. If your notebook uses a different role (via `USE ROLE`), CoCo's metadata visibility may differ. The `USE ROLE` prefix in the prompt above aligns CoCo's session to match. When results differ, treat the cell as the source of truth.

> 💡 **Helping CoCo find your objects.** CoCo's context defaults to the User Defaults that we set in the Environment Setup lab. This is (user)_RL, (user)_WH, (user)_DB, and (user)_DB.RAW. When creating objects, CoCo may not find them unless you tell it where to look. Give CoCo the **full path** — like "analyze the table `MY_DB.PUBLIC.ORDERS`" or "@MY_DB.PUBLIC write SQL to…" — so it can locate your data."

### CoCo Panel Icons.

When CoCo generates code, you will notice several icons in the code block header:

![CoCo block header (image)](images/pp_coco_code_block_header.png)

- **+ (Insert)** — Inserts the generated code into a new notebook cell
- **Replace** — Replaces the currently selected notebook cell with the generated code
- **Expand** — Expands the code block for easier viewing and copying
- **Copy** - Copy the code block

:warning: **For this lab:** To keep things simple, we recommend **copying and pasting** the generated code manually rather than using these buttons. This gives you more control over where code is placed and avoids accidentally overwriting existing cells.





---

:point_right: **Go ahead and open the CoCo panel now.** We will use it throughout this lab.

## Set Up Your Workspace
First, set your **personal context**. The cell below switches to your own role, creates and uses your own warehouse, then points reads at the shared, read-only course data. Your warehouse is named after your username, so everyone in the account stays out of each other's way.

In [ ]:
%%sql -r dataframe_1
-- Set your personal context: your role + your own warehouse
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
-- Read from the shared, read-only course data
USE DATABASE PLATPLUS_DB;
USE SCHEMA ROASTERY_SALES;

## 1 · Ask in English, Get SQL

The simplest CoCo workflow: ask a business question in plain English, read the SQL it generates, paste it into a notebook cell, and run it.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">
What were total roastery sales in the last 30 days? Generate SQL to show me.
</pre>

After pasting, press **Enter** (or click the **send** icon) to run the prompt.

> 📌 **If CoCo runs the SQL for you** instead of printing it as a code block, expand the query/tool step in CoCo's response to reveal the SQL, then use the **Copy** button — or just ask CoCo "show me the SQL". You'll reuse this trick throughout the lab whenever a cell asks you to paste CoCo's SQL.

> 👉 **Next steps:**
> 1. CoCo will work through a sequence of steps in order to generate appropriate SQL. Are these surprising?
> 2. Copy the generated SQL from the CoCo panel using the _Copy code to clipboard_ button.
> 3. Paste it into the empty cell below and run it — **read it before you run it**.

In [ ]:
%%sql -r dataframe_2
-- Paste CoCo's generated SQL on the blank line below, then run it


> 🔎 **What to expect:** One row, one number — the total revenue from all orders in the last 30 days. `SUM(total_amount)` collapses all order rows into a single figure; the `WHERE` clause narrows to the time window.

## 2 · When CoCo Needs More Context

CoCo can only answer business questions that it can ground in your actual data. Two things can trip it up: **vague questions** and **missing schema context**. This section covers both.

---

### Part A — Vague questions trigger clarification

When your question is ambiguous, CoCo may pause and ask you to clarify before it writes any SQL.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">
What are the best of them?
</pre>

> 👉 **Notice CoCo's response.** "best" is undefined and "them" has no clear referent. Often CoCo will pause and ask you to clarify — for example, *"Best of what? Are you looking for top stores, top products, or top customers?"* Sometimes, using earlier context in the chat, it will make a reasonable assumption and answer anyway — but that guess won't always match what you meant.
>
> **Takeaway:** Write prompts with a clear subject and a measurable definition of success (e.g., "top 5 stores **by sales revenue**").

---

### Part B — Anchoring CoCo to a schema with `@`

Even a well-worded question can miss the mark if CoCo doesn't know *where* your data lives. The `@` mention pins CoCo to a specific database, schema, or table so it can see the real columns and generate accurate SQL.

Type `@` in the CoCo prompt to open the object picker. Navigate to `PLATPLUS_DB → ROASTERY_SALES` and select it. This immediately grounds CoCo in your actual schema so it can see what tables and columns are available.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">
@PLATPLUS_DB.ROASTERY_SALES What are the top five stores by sales revenue?
</pre>

> 👉 **CoCo now has the context it needs.** It can see the available tables, pick the right one, and use actual column names. 

Copy and paste the generated SQL into the empty cell below, **remove the** `LIMIT` **clause to bring back all results**, and run it. *(If CoCo ran it inline, expand its step to reveal and copy the SQL.)*

In [ ]:
%%sql -r dataframe_3
-- Paste CoCo's anchored SQL on the blank line below, then run it


### Check your understanding 🧠
Which store came out on top in the revenue results?

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q4")

---

> 🔎 **The lesson:** Vague questions produce vague answers. The `@` reference is how you give CoCo the schema context it needs to generate accurate, runnable SQL rather than guessing at table and column names.

---

### Exploring with `@`

The `@` picker lets you browse your entire data catalog from inside the CoCo prompt:

| What you type | What CoCo can see |
|---|---|
| `@DATABASE` | All schemas in that database |
| `@DATABASE.SCHEMA` | All tables, views, and objects in the schema |
| `@DATABASE.SCHEMA.TABLE` | Column names and types for a specific table |

**Try it in two steps:**

🤖 **CoCo Prompt 1** — *click to copy, then paste into the CoCo panel, and do not hit return/enter:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">@PLATPLUS_DB.ROASTERY_SALES</pre>

Select a table from the popup that appears, and then hit return/enter to browse its columns and data.

🤖 **CoCo Prompt 2** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">@PLATPLUS_DB.ROASTERY_SALES What tables are in this schema and what does each one contain?</pre>

## 3 · Iterate: Make It Yours
In the real world analysts rarely start from a blank prompt. They take a query they have and ask for the next thing. This is that pattern: hand CoCo an existing query and a natural-language extension, and let it build from there.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">I have this query that shows total Roastery sales over the last 30 days. Extend it to: (1) filter to the last 90 days instead of 30, (2) break it down by store name, and (3) add a column showing each store's percentage share of total revenue in the period.

SELECT ROUND(SUM(o.total_amount), 2) AS revenue
FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS o
WHERE o.order_ts >= DATEADD(day, -30, CURRENT_TIMESTAMP());</pre>

In [ ]:
%%sql -r dataframe_4
-- Paste CoCo's extended query on the blank line below, then run it


> 🔎 **Review the output.** The revenue share column uses a window function — `SUM(revenue) OVER ()` computes the total revenue across all stores, so each row's share is `revenue / SUM(revenue) OVER ()`. Read CoCo's explanation of that clause — it's worth understanding before you use it elsewhere.

> 💡 **Your turn:** Ask CoCo one more extension:

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Now filter to only show stores where revenue share is above 20%.</pre>

Notice how filtering on a window-function result requires a different approach than a normal `WHERE`.

## 4 · CoCo Does the Research

You know how to aggregate numbers. But what about the *actual words* customers wrote? The `FEEDBACK` table has a `comment` column — free text. Aggregating text is a different problem, and it's a good example of using CoCo as a research partner: ask it what Snowflake can do, not just what the data says.

**Ask CoCo how to surface the text:**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">
@PLATPLUS_DB.ROASTERY_SALES.FEEDBACK I want to see the actual customer comment text for stores with low scores — all comments grouped by store in a single row. What Snowflake function do I need? Link me to the documentation.
</pre>

> 👉 **What to watch for:** CoCo should surface `LISTAGG` — a Snowflake aggregate function that concatenates text values into a single delimited string per group. It will also point you to the [Snowflake aggregate functions reference](https://docs.snowflake.com/en/sql-reference/functions-aggregation). **Read the explanation before running anything.**

> 💡 **Why this matters:** Most SQL aggregates work on numbers (`SUM`, `AVG`, `COUNT`). `LISTAGG` does the same thing for text — it collapses multiple rows of comments into one readable string per store. This pattern turns raw feedback rows into a scannable summary.

In [ ]:
%%sql -r dataframe_5
-- CoCo's suggested SQL — paste it here, or use the version below as a reference
SELECT s.store_name,
       ROUND(AVG(f.score), 2)                                          AS avg_score,
       COUNT(f.feedback_id)                                            AS responses,
       LISTAGG(f.comment, ' | ')
           WITHIN GROUP (ORDER BY f.feedback_date DESC)                AS recent_comments
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK f
JOIN PLATPLUS_DB.ROASTERY_SALES.STORES   s ON s.store_id = f.store_id
WHERE f.score < 4
GROUP BY s.store_name
HAVING COUNT(f.feedback_id) >= 2
ORDER BY avg_score ASC;

> 🔎 **Review the output.** Each row is one store. `recent_comments` shows all feedback text for that store joined with ` | ` separators — scroll right to read them. Notice `WITHIN GROUP (ORDER BY f.feedback_date DESC)` controls the order within the concatenated string.
>
> 📌 CoCo's own version may differ (e.g. it might filter with `WHERE score <= 2` and skip `HAVING`). If it does, use the reference query in the cell above so the `HAVING` explanation below matches what you actually ran.

> 🧠 **Spot the new clause:** The query uses `HAVING COUNT(f.feedback_id) >= 2` — not `WHERE`. 
> - **`WHERE`** filters individual rows *before* grouping — it can't see aggregate results like `COUNT(*)`.
> - **`HAVING`** filters groups *after* `GROUP BY` runs — it can see `COUNT(*)`, `AVG(score)`, and any other aggregate.
> 
> Asking for "stores with at least 2 responses" is always a `HAVING` question because the count is computed per group.

---

**Go further — ask CoCo about Snowflake's AI layer:**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Does Snowflake have a built-in AI SQL function for sentiment analyis? Use the latest function. Show me how to use it on the comment column in PLATPLUS_DB.ROASTERY_SALES.FEEDBACK — and what does it return?</pre>

> 🔎 **What to expect:** CoCo confirms that the function `AI_SENTIMENT` exists, provides documentation links, and shows how to call it. Note that it returns a **structured object**, not a simple number — the overall sentiment category (positive / negative / neutral / mixed) lives at `['categories'][0]['sentiment']`. You don't need to memorise that path today.
>
> 📌 **Coming up in the AI Functions lab.** For now the point is that Snowflake's AI layer can analyse text directly in SQL — no external tools, no data export, no Python required.

## 5 · Skills: CoCo With a Structured Workflow

> 📌 **Skills ship with Snowflake.** The built-in skills in this section are part of the Snowflake environment — available directly through CoCo in Snowsight, with nothing to install or configure. Snowflake adds new ones over time.

Out of the box, CoCo is already capable: it combines broad trained knowledge of SQL and Snowflake with **live tools it can call on** — including searching the current Snowflake documentation and inspecting the actual objects in your account. What it *doesn't* have by default is a fixed **method**. For each task it decides how to approach the problem on the fly, so the steps — and the results — can vary from one run to the next.

**A skill changes this.** A skill injects a specific, curated *workflow* into the conversation — not just knowledge, but *how to approach a problem*, step by step, in your context. The `/sql-author` skill doesn't just know SQL: it follows a prescribed sequence — read the error, inspect the real schema, fix the root cause, validate before presenting. The `/warehouse` skill doesn't just know about warehouses: it looks up your specific warehouse, identifies its generation, and applies the right advice.

Skills are also **repeatable and shareable**: every student who invokes `/sql-author` gets the same structured diagnostic approach. That's why CoCo shows `Using skill: skill-name` in its response — it's signalling: *"I've loaded a specific workflow for this task, not just free-form reasoning."*

**How to invoke a skill:**
- Type `/` in the CoCo prompt to open the skill menu — you'll see Built-in (Snowflake-provided) and Local (workspace) skills
- Or type `/<skill-name>` directly, followed by your question
- **Always confirm activation:** look for the `Using skill:` line in CoCo's first response

### Discover what's available

Start by listing every skill CoCo can see in this session.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">
/skill list
</pre>

> 👉 **Scan the output.** You'll see built-in Snowflake skills (covering warehouses, SQL, dynamic tables, data governance, AI functions, and more) and any local skills in this workspace. The list evolves as Snowflake ships new skills, so what you see may differ from the docs.

### Try it: `/warehouse`

The `warehouse` skill specialises in Snowflake compute: sizing, generations (Gen1 vs Gen2 vs Adaptive), credits, performance tuning, and DDL. Ask CoCo to elaborate on the skills capabilities.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">
/warehouse what does this skill do?
</pre>

Now ask a real question — one about the warehouse you just created.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">
/warehouse I created an XSMALL warehouse with AUTO_SUSPEND=60 and AUTO_RESUME=TRUE. I'm running SQL analytics queries that join a few tables with a couple of thousand rows each. Is this the right size and configuration for this kind of workload? What does it cost per hour?
</pre>

> 👉 **Look for `Using skill: warehouse`** at the top of CoCo's response before it answers. That line confirms the skill is active — CoCo is now following the warehouse skill's structured workflow (look up your warehouse, identify its generation, apply the right advice) rather than reasoning freeform.

> 🔎 **What to expect:** The skill will confirm XSMALL is appropriate for small analytical workloads, validate the AUTO_SUSPEND and AUTO_RESUME settings as correct governance, and give you the Gen2 credit rate. You just validated the setup from the lab's opening cell using a skill that knows the right questions to ask.

### Try it: `/sql-author`

The `sql-author` skill is for writing, fixing, and debugging Snowflake SQL. But let's ask CoCo to provide detail on its capabilities.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">
/sql-author what does this skill do?
</pre>

This skill's workflow is deliberate: it reads the full intent first, then inspects the *real* schema (it runs `DESCRIBE TABLE` to see actual column names — not guesses), fixes the root cause rather than just the first error, and validates the corrected SQL before presenting it.

The query below has **four bugs** — all realistic mistakes a new SQL author might write. Copy the full block into the CoCo prompt and let the skill diagnose it.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">
/sql-author This query fails — please diagnose and fix every bug:

SELECT s.store_name,
       COUNT(*) AS total_orders,
       SUM(o.total_amount) AS revenue,
       AVG(f.score) AS satisfaction
FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS o
JOIN PLATPLUS_DB.ROASTERY_SALES.STORES s ON o.id = s.store_id
JOIN PLATPLUS_DB.ROASTERY_SALES.FEEDBACK f ON f.store = s.store_id
WHERE revenue > 5000
GROUP BY store_name
ORDER BY revenue DESC;
</pre>

> 👉 **Look for `Using skill: sql-author`** before CoCo starts its analysis. Then watch how it works: it inspects the actual table columns first (rather than guessing), identifies each bug, explains *why* it's wrong, and validates the corrected query before presenting it.

> 🔎 **The four bugs — see if CoCo catches all of them:**
> 1. `o.id = s.store_id` — ORDERS doesn't have an `id` column; the join key is `o.store_id`
> 2. `f.store = s.store_id` — FEEDBACK's join column is `f.store_id`, not `f.store`
> 3. `WHERE revenue > 5000` — `revenue` is an alias computed by `SUM()`, which doesn't exist at the `WHERE` stage; this needs to move to `HAVING SUM(o.total_amount) > 5000`
> 4. `GROUP BY store_name` — ambiguous; should be `GROUP BY s.store_name`
>
> ⚠️ **Heads up:** CoCo often fixes bug #4 silently — rewriting `GROUP BY store_name` to `s.store_name` without listing it — so it may report only **3** bugs. Check the corrected query to confirm all four are handled.

> 💡 **Why not just ask CoCo without the skill?** You can — and it will probably get most of them. The difference: the skill's workflow ensures it *inspects the real schema* before guessing, *validates the fix before showing you*, and catches the semantically wrong SQL (bug #3) rather than just the syntax errors. Without the skill, you might get a confidently presented fix that still fails.

### Your own skill: `roastery-analyst`

Built-in skills cover Snowflake's own features. **Custom skills** are yours — you encode the domain knowledge specific to your data and your team's questions, so CoCo doesn't have to rediscover it every session.

**This lab includes the `roastery-analyst` skill** in the `roastery-analyst-skill/` folder. It knows the full `ROASTERY_SALES` schema, business context (6 stores across 3 regions, what "store health" means in this business), and the common analytical patterns for this dataset.

**See the difference — try this without the skill first.**

> 👉 **Open a brand new chat first.** In the CoCo panel, click the **new chat icon** (the **+** at the top of the panel). This starts a completely fresh conversation with no prior context. It's important that the skill is *not* already loaded when you send this first prompt. We will also provide explicit directives to CoCo in our prompt below to dissuade it from using skills to provide an answer in this instance.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">
Which of our stores needs the most attention right now? Don't use skills to provide this answer.
</pre>

> CoCo will ask clarifying questions — it doesn't know what "our stores" means or what "attention" looks like in your data.

**Upload the skill to CoCo in three steps:**
1. In the notebook files panel, open the **`roastery-analyst-skill/`** folder and download **`SKILL.md`** to your computer
2. In the CoCo panel, click the **`+` button in the message box** (not the one at the top of the panel) → select **Upload Skill File(s)** → choose the `SKILL.md` file you just downloaded
3. CoCo confirms the skill is uploaded — you'll see it listed under Local skills

**Now invoke the skill and ask the same question.**

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/roastery-analyst Which of our stores needs the most attention right now?</pre>

> 🔎 **Read the response carefully — it’s a report, not just a query.** Because the skill knows the business context, CoCo doesn’t just generate SQL — it analyses the output and gives you a structured narrative. Look for:
> - Which stores are flagged and **why** (revenue trend and satisfaction score read together, not in isolation)
> - Specific numbers — store names, average scores, revenue figures with context
> - Any caveats about data volume or recency
> - Recommended next actions
>
> This is what a custom skill unlocks: instead of answering “what tables do you have?”, CoCo can answer “what should I do about my business?” — because it already knows the domain. No `@` reference needed.

In [ ]:
%%sql -r dataframe_6
-- Paste the skill-powered query on the blank line below, then run it
-- The skill often runs several queries and returns a report. Use the store-health
-- query (revenue + satisfaction per store). If CoCo ran it inline, expand that step
-- in CoCo's response to reveal the SQL, then copy it here.


> 🔎 **The lesson:** Skills sit on a spectrum from built-in (Snowflake-provided, updated with the product) to custom (you build once, the team uses everywhere). The right skill for the right task means CoCo is following a proven workflow — not improvising. That makes its output more predictable, more debuggable, and more useful as a team asset.

## 6 · Trust, but Verify

CoCo gives confident answers — but confidence isn’t the same as correctness. One habit separates reliable analysts from fast ones: **ask CoCo to explain its assumptions, then check them against the actual data.**

Here is a question that sounds simple. Ask it, read the answer, then pause before you use it.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">How many orders has the Roastery received in the last 30 days?</pre>

CoCo will give you a number, using a rolling 30-day window from today’s date. Before you use it, answer one question: **is “last 30 days” even a meaningful window for this dataset?**

In [ ]:
%%sql -r dataframe_7
SELECT MIN(order_ts) AS earliest_order,
       MAX(order_ts) AS latest_order,
       DATEDIFF(day, MIN(order_ts), MAX(order_ts)) AS data_span_days
FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS;


> 💡 **The verify habit:** CoCo can’t know your data’s temporal coverage unless you tell it. Before using any time-windowed result in a report, always run a quick date-range check. CoCo’s job is to write the query; yours is to make sure the question is answerable with the data you actually have.

## 7 · Take It to Your Role
Same skills, your kind of work. Run the section that matches your role — skim the others to see how the same patterns travel.

### Data Analyst — surface the feedback story

Numbers tell you *what*. Comments tell you *why*. Use `LISTAGG` to pull the feedback narrative for the region with the lowest average score — the kind of thing you'd put in a slide before a store review meeting.

> 📌 **Make sure the `roastery-analyst` skill is loaded** before running this prompt — it gives CoCo the Roastery schema context it needs.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">/roastery-analyst Show me the average feedback score and all customer comments grouped by region, worst region first.</pre>

In [ ]:
%%sql -r dataframe_8
SELECT s.region,
       ROUND(AVG(f.score), 2)                                            AS avg_score,
       COUNT(f.feedback_id)                                              AS responses,
       LISTAGG(f.comment, ' | ')
           WITHIN GROUP (ORDER BY f.feedback_date DESC)                  AS comments
FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK f
JOIN PLATPLUS_DB.ROASTERY_SALES.STORES   s ON s.store_id = f.store_id
GROUP BY s.region
ORDER BY avg_score ASC;

> 🔎 **Review the output.** Which region has the lowest score, and what are customers actually saying? This query gives you the evidence you need for a recommendation — not just a number, but the language behind it.

### Data Engineer — build something that stays fresh

A query answers a question once. A [**Dynamic Table**](https://docs.snowflake.com/en/user-guide/dynamic-tables/overview) answers it continuously — the SQL runs on a schedule, the result persists as a table, and anyone who queries it always gets current data. No cron job, no manual refresh.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">Give me a brief explanation of what Snowflake dynamic tables are, provide the documentation links, and reasons why the TARGET_LAG parameter is crucial.
Then turn my month-over-month revenue query into a dynamic table named `MOM_REVENUE` in my own database ({{my_db}}) and `LAB` schema that refreshes every hour. Provide me the SQL so that I can review and create it.</pre>

The cell below sets up your personal workspace for this exercise.

In [ ]:
%%sql -r dataframe_9
-- Create your personal database and schema if they don't exist yet
USE ROLE {{my_role}};
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA  IF NOT EXISTS {{my_db}}.LAB;
USE {{my_db}}.LAB;

> 📘 **What is `TARGET_LAG`?** It tells Snowflake the *maximum acceptable staleness* for the dynamic table's data. `TARGET_LAG = '1 hour'` means: "recompute the underlying query whenever the result is more than one hour old." Snowflake handles the scheduling — you just declare how fresh you need it.

> 👉 **Paste CoCo's `CREATE DYNAMIC TABLE` statement into the cell below**, then run it. *(Expand CoCo's step to copy the SQL if it ran inline. You may need to iterate with CoCo a couple of times to get a version that runs cleanly — that back-and-forth is expected.)*

In [ ]:
%%sql -r dataframe_10
-- Paste CoCo's CREATE DYNAMIC TABLE on the blank line below, or use the reference version in the next cell


In [ ]:
%%sql -r dataframe_11
-- Reference version
CREATE OR REPLACE DYNAMIC TABLE {{user}}_DB.LAB.MOM_REVENUE
    TARGET_LAG = '1 hour'
    WAREHOUSE = {{my_wh}}
    REFRESH_MODE = AUTO
AS
    SELECT
        DATE_TRUNC('month', o.order_ts)::DATE AS revenue_month,
        s.region,
        s.store_name,
        ROUND(SUM(o.total_amount), 2) AS monthly_revenue,
        LAG(SUM(o.total_amount)) OVER (
            PARTITION BY s.store_id
            ORDER BY DATE_TRUNC('month', o.order_ts)
        ) AS prev_month_revenue,
        ROUND(
            (SUM(o.total_amount) - LAG(SUM(o.total_amount)) OVER (
                PARTITION BY s.store_id
                ORDER BY DATE_TRUNC('month', o.order_ts)
            )) / NULLIF(LAG(SUM(o.total_amount)) OVER (
                PARTITION BY s.store_id
                ORDER BY DATE_TRUNC('month', o.order_ts)
            ), 0) * 100,
        2) AS mom_pct_change
    FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS o
    JOIN PLATPLUS_DB.ROASTERY_SALES.STORES s ON s.store_id = o.store_id
    GROUP BY s.store_id, s.store_name, s.region, DATE_TRUNC('month', o.order_ts);


> 👉 **Query your dynamic table.** It behaves exactly like a regular table — no special syntax needed.

In [ ]:
%%sql -r dataframe_12
SELECT * FROM {{my_db}}.LAB.MOM_REVENUE
ORDER BY store_name, revenue_month;

> 🔎 **Review the output.** Any store–month combinations shown are months where that store's revenue *fell* compared to the previous month. The dynamic table will refresh automatically — next time you query it, the underlying query has re-run in the background.

> 💡 **Clean up:** Run the cell below to drop the dynamic table. The `{{my_db}}.LAB` schema stays in place — you'll use it in Lab 3.

In [ ]:
%%sql -r dataframe_13
DROP DYNAMIC TABLE IF EXISTS {{my_db}}.LAB.MOM_REVENUE;

### Platform Admin — know what you have before you grant

Before deciding who gets access to what, it helps to understand what the data looks like — specifically which columns might contain sensitive information. `INFORMATION_SCHEMA` lets you inspect column metadata without reading the data itself.

🤖 **CoCo Prompt** — *click to copy, then paste into the CoCo panel:*

<pre style="user-select: all; -webkit-user-select: all; background: rgba(41, 181, 232, 0.12); border-left: 4px solid #29B5E8; padding: 12px; border-radius: 0 4px 4px 0; cursor: pointer; white-space: pre-wrap; word-wrap: break-word;">First run USE ROLE {{my_role}}, then using INFORMATION_SCHEMA show me all columns in PLATPLUS_DB.ROASTERY_SALES and flag any that might contain personal or sensitive data based on their names.</pre>

> ⚠️ **Why the prompt says "USE ROLE" first:** CoCo's panel may run under a different role than the one active in your notebook. By explicitly telling CoCo to switch to `{{my_role}}`, you ensure it has the same metadata visibility as your notebook session.

The pattern: inspect metadata first, then design grants around what you find. In the next lab you'll do exactly this — grant precisely the columns and tables that each role needs, nothing more.

In [ ]:
%%sql -r dataframe_14
SELECT table_name,
       column_name,
       data_type
FROM PLATPLUS_DB.INFORMATION_SCHEMA.COLUMNS
WHERE table_schema = 'ROASTERY_SALES'
ORDER BY table_name, ordinal_position;

> 🔎 **Review the output.** Which columns stand out as potentially sensitive? Which did CoCo highlight in its summary of concerns?

- `CUSTOMERS.FULL_NAME` (personally identifiable — a customer's name)
- `FEEDBACK.COMMENT` (free text — could contain names, complaints)
- `FEEDBACK.CUSTOMER_ID` / `ORDERS.CUSTOMER_ID` (indirect identifiers — join keys that link to PII in the CUSTOMERS table)

In Lab 3 you'll decide who gets `SELECT` on these and design the grant chain around that decision.

> ⚠️ **CoCo uses your workspace session role — the cell uses your notebook role.** CoCo's panel runs under the role shown in your workspace context, which may not match the `USE ROLE` set inside this notebook. If CoCo missed a table (like CUSTOMERS), it's because that panel role lacks access to its metadata. The SQL cell above runs under `{{my_role}}` and shows the complete picture. When results differ, **treat the cell as the source of truth** — then use CoCo to help interpret what you found.

## Check Your Work
Run the next cell to **check your work**. It confirms you can reach and query the shared course data with your own role and warehouse, and prints a ✅ / ⛔ result for each check — no need to read the code.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
checks = [
    ("You can reach the shared store data", "SELECT COUNT(*) FROM PLATPLUS_DB.ROASTERY_SALES.STORES", 1),
    ("Orders are queryable for analysis", "SELECT COUNT(*) FROM PLATPLUS_DB.ROASTERY_SALES.ORDERS", 1),
    ("Feedback is queryable for analysis", "SELECT COUNT(*) FROM PLATPLUS_DB.ROASTERY_SALES.FEEDBACK", 1),
]
passed = 0
for label, sql, need in checks:
    try:
        n = session.sql(sql).collect()[0][0]
        ok = n >= need
    except Exception:
        n, ok = None, False
    passed += 1 if ok else 0
    icon = '✅' if ok else '⛔'
    detail = f'(found {n})' if n is not None else '(not found)'
    print(f'{icon}  {label}  {detail}')
print('-' * 52)
if passed == len(checks):
    print(f'🎉 ALL {len(checks)} CHECKS PASSED — lab complete! Nice work.')
else:
    print(f'⚠️  {passed}/{len(checks)} passed — re-run the cells above for any ⛔ item, then run this cell again.')

### Check your understanding 🧠
A few quick questions on this lab's key themes. Run each cell below, choose an answer, and the feedback appears inline — no grade, just a self-check.

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q5")

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q6")

In [ ]:
from resources.knowledgecheck import quiz
quiz.display_question("q7")

### Key takeaways. ❄️
- CoCo turns plain-English questions into SQL — but you read, run, and verify it
- The `@` picker grounds CoCo in your actual schema; a skill grounds it in your business domain
- `WHERE` filters rows before grouping; `HAVING` filters groups after — they are not interchangeable
- `LISTAGG` aggregates text the same way `SUM` aggregates numbers — useful for surfacing qualitative data
- Dynamic tables persist a query's result and refresh it automatically — engineers build once, analysts query forever

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.